# HIBA sensitivity analysis by image type (dermoscopic vs clinical photographs)

Follows `data/derm/hiba_image_type_plan.json`. Fetches HIBA image-type labels from the ISIC archive (metadata only), then repeats the main skin-cancer analyses separately for dermoscopic and clinical images: model AUCs, the second-reader benefit at matched volume (H1), and the training-data contrast (H3). CPU, about 10 minutes.

In [1]:
# Cell 1: load predictions and fetch image types
from google.colab import drive
drive.mount('/content/drive')
import os, json, re, time, datetime, itertools, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import requests, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
ARCHS = ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2']; SOURCES = ['HAM10000', 'BCN20000']
MODELS = [(s, a, k) for s in SOURCES for a in ARCHS for k in (0, 1)]; mname = lambda s, a, k: f'{s}_{a}_seed{k}'
VARIANTS = [(s, a) for s in SOURCES for a in ARCHS]
D = pd.read_csv(f'{DERM}/manifest_HIBA.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True)
S = requests.Session(); rows, url, params = [], 'https://api.isic-archive.com/api/v2/images/search/', {'collections': 251, 'limit': 100}
def flatten(d, prefix=''):
    out = {}
    for k, v in (d or {}).items():
        if isinstance(v, dict): out.update(flatten(v, f'{prefix}{k}.'))
        else: out[f'{prefix}{k}'] = v
    return out
while url:
    for i in range(5):
        try: r = S.get(url, params=params, timeout=60); r.raise_for_status(); j = r.json(); break
        except Exception:
            if i == 4: raise
            time.sleep(2 * (i + 1))
    params = None; url = j.get('next')
    rows += [{'image_id': x['isic_id'], **flatten(x.get('metadata'), 'm.')} for x in j['results']]
M = pd.DataFrame(rows); tcol = next(c for c in M.columns if c.endswith('image_type'))
D = D.merge(M[['image_id', tcol]].rename(columns={tcol: 'image_type'}), on='image_id', how='left')
D['group'] = np.where(D.image_type.astype(str).str.lower().str.startswith('dermoscop'), 'dermoscopic',
                      np.where(D.image_type.astype(str).str.lower().str.startswith('clinical'), 'clinical', 'unknown'))
print(D.image_type.value_counts(dropna=False).to_string()); print(D.groupby('group').agg(images=('image_id', 'size'), patients=('patient_id', 'nunique'),
      melanoma=('dx', lambda x: int((x == 'MEL').sum())), cancer=('dx', lambda x: int(x.isin(['MEL', 'BCC', 'SCC']).sum()))).to_string())
P = {m: pd.read_csv(f'{DERM}/predictions/HIBA__{mname(*m)}.csv', dtype={'image_id': str}).set_index('image_id').p_malignant.reindex(D.image_id).to_numpy() for m in MODELS}

Mounted at /content/drive
image_type
dermoscopic           1270
clinical: overview     340
clinical: close-up       6
             images  patients  melanoma  cancer
group                                          
clinical        346       244        59     218
dermoscopic    1270       584       194     533


In [2]:
# Cell 2: per group, AUCs, H1 (matched-volume reduction) and H3 (training-data contrast), patient bootstrap
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def combos(va, vb):
    if va == vb: return [((*va, 0), (*vb, 1))]
    return [((*va, i), (*vb, j)) for i in (0, 1) for j in (0, 1)]
def ci(v): return [round(float(np.nanpercentile(v, 2.5)), 4), round(float(np.nanpercentile(v, 97.5)), 4)]
RNG = np.random.default_rng(20261605); OUT = {}
for grp in ('dermoscopic', 'clinical'):
    idx = np.flatnonzero(D.group.to_numpy() == grp); d = D.iloc[idx].reset_index(drop=True); n = len(d)
    mal = d.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy(); mel = d.dx.eq('MEL').to_numpy(); f = 1.31 * mal.mean()
    p = {m: P[m][idx] for m in MODELS}; R = {m: refer(p[m], f) for m in MODELS}; RK = {m: rankdata(p[m]) / n for m in MODELS}
    cl = pd.factorize(d.patient_id.astype(str))[0]; ncl = cl.max() + 1
    W = [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[cl].astype(float) for _ in range(1000)]
    res = {'images': n, 'patients': int(ncl), 'melanoma': int(mel.sum()), 'cancer': int(mal.sum()), 'referral_share': round(f, 4),
           'auc_median': round(float(np.median([roc_auc_score(mal, p[m]) for m in MODELS])), 4),
           'auc_range': [round(float(min(roc_auc_score(mal, p[m]) for m in MODELS)), 4), round(float(max(roc_auc_score(mal, p[m]) for m in MODELS)), 4)]}
    for tname, y in (('melanoma', mel), ('any cancer', mal)):
        anc = ens = 0
        for va in VARIANTS:
            for vb in VARIANTS:
                if va[1] == vb[1]: continue
                for ka, kb in combos(va, vb):
                    anc = anc + (y & ~R[ka]); ens = ens + (y & ~refer(np.maximum(RK[ka], RK[kb]), f))
        red = lambda w: (anc @ w - ens @ w) / (anc @ w) if anc @ w > 0 else np.nan
        res[f'H1_{tname}'] = {'relative_reduction': round(float(red(np.ones(n))), 4), 'ci95': ci([red(w) for w in W])}
        VEC = {}
        for i_, va in enumerate(VARIANTS):
            for vb in VARIANTS[i_:]:
                if va[1] == vb[1]: continue
                cs = combos(va, vb); ma = mb = both = 0
                for ka, kb in cs:
                    x, z = y & ~R[ka], y & ~R[kb]; ma = ma + x; mb = mb + z; both = both + (x & z)
                VEC[(va, vb)] = (ma / len(cs), mb / len(cs), both / len(cs), va[0] == vb[0])
        def contrast(w):
            T = y @ w; same, diff = [], []
            for (ma, mb, both, ss) in VEC.values():
                a_, b_, c_ = ma @ w, mb @ w, both @ w
                if a_ * b_ > 0: (same if ss else diff).append(c_ / (a_ * b_ / T))
            return np.mean(same) - np.mean(diff), np.mean(same), np.mean(diff)
        obs = contrast(np.ones(n)); bs = [contrast(w)[0] for w in W]
        res[f'H3_{tname}'] = {'same_data_ratio': round(float(obs[1]), 3), 'different_data_ratio': round(float(obs[2]), 3),
                              'same_minus_different': round(float(obs[0]), 3), 'ci95': ci(bs)}
    OUT[grp] = res; print(grp, json.dumps(res, indent=1))
json.dump({'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'hiba_image_type_plan.json', 'status': 'exploratory sensitivity',
           'image_type_counts': D.image_type.value_counts(dropna=False).to_dict(), 'results': OUT}, open(f'{DERM}/hiba_image_type_results.json', 'w'), indent=2, default=str)
print('Wrote data/derm/hiba_image_type_results.json')

dermoscopic {
 "images": 1270,
 "patients": 584,
 "melanoma": 194,
 "cancer": 533,
 "referral_share": 0.5498,
 "auc_median": 0.9064,
 "auc_range": [
  0.8722,
  0.9357
 ],
 "H1_melanoma": {
  "relative_reduction": 0.1153,
  "ci95": [
   0.0582,
   0.191
  ]
 },
 "H3_melanoma": {
  "same_data_ratio": 4.104,
  "different_data_ratio": 3.531,
  "same_minus_different": 0.573,
  "ci95": [
   0.2257,
   1.1622
  ]
 },
 "H1_any cancer": {
  "relative_reduction": 0.2129,
  "ci95": [
   0.1576,
   0.2766
  ]
 },
 "H3_any cancer": {
  "same_data_ratio": 5.203,
  "different_data_ratio": 4.164,
  "same_minus_different": 1.039,
  "ci95": [
   0.5935,
   1.6682
  ]
 }
}
clinical {
 "images": 346,
 "patients": 244,
 "melanoma": 59,
 "cancer": 218,
 "referral_share": 0.8254,
 "auc_median": 0.7234,
 "auc_range": [
  0.6055,
  0.8166
 ],
 "H1_melanoma": {
  "relative_reduction": 0.0516,
  "ci95": [
   -0.0441,
   0.155
  ]
 },
 "H3_melanoma": {
  "same_data_ratio": 2.225,
  "different_data_ratio": 1.985,